# ACE RAG — 01 Indexing

Run from the project root. This notebook generates document embeddings through the deployed Modal Qwen3-Embedding-8B service, checkpoints them locally, and upserts them into the existing persistent Chroma service.

**Storage:** Modal Volume is the live persistent database. `embedded_chunks.jsonl` is a resumable local embedding artifact. Hugging Face (`emitev/ace-rag-data`) stores versioned backups, not live writes.

In [1]:
import json, os, math
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import modal

INPUT_FILE = Path("data/enriched_chunks.jsonl")
EMBEDDED_FILE = Path("data/embedded_chunks.jsonl")

EMBEDDING_APP_NAME = "ace-rag"
EMBEDDING_CLASS_NAME = "EmbeddingService"

CHUNKS_PER_BATCH = 100
MODAL_REQUEST_BATCH_SIZE = 16
MODEL_INFERENCE_BATCH_SIZE = 4
EXPECTED_EMBEDDING_DIMENSION = 4096

HF_REPO_ID = "emitev/ace-rag-data"
HF_REPO_TYPE = "model"

assert INPUT_FILE.exists(), f"Missing input: {INPUT_FILE}"
EMBEDDED_FILE.parent.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_FILE)

Input: data\enriched_chunks.jsonl


In [2]:
def load_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON at {path}, line {line_no}") from exc
    return records

chunks = load_jsonl(INPUT_FILE)
required = {"id", "text", "metadata"}
ids = set()

for i, chunk in enumerate(chunks):
    missing = required - chunk.keys()
    if missing:
        raise ValueError(f"Chunk {i} missing fields: {missing}")
    if not isinstance(chunk["id"], str) or not chunk["id"]:
        raise ValueError(f"Chunk {i} has invalid ID")
    if not isinstance(chunk["text"], str) or not chunk["text"].strip():
        raise ValueError(f"Chunk {chunk['id']} has empty text")
    if not isinstance(chunk["metadata"], dict):
        raise ValueError(f"Chunk {chunk['id']} has invalid metadata")
    if chunk["id"] in ids:
        raise ValueError(f"Duplicate source ID: {chunk['id']}")
    ids.add(chunk["id"])

print(f"Loaded {len(chunks):,} chunks")

Loaded 3,464 chunks


In [3]:
EmbeddingServiceRemote = modal.Cls.from_name(
    EMBEDDING_APP_NAME,
    EMBEDDING_CLASS_NAME,
)
embedding_service = EmbeddingServiceRemote()
print("Connected to deployed embedding service.")

Connected to deployed embedding service.


In [4]:
def load_embedding_checkpoint(path):
    existing = {}
    if not Path(path).exists():
        return existing

    with Path(path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid checkpoint JSON at line {line_no}. "
                    "Repair/remove the incomplete line before resuming."
                ) from exc
            if "id" not in record or "embedding" not in record:
                raise ValueError(f"Checkpoint line {line_no} lacks id/embedding")
            existing[record["id"]] = record
    return existing


def validate_embedding(vector):
    vector = np.asarray(vector, dtype=np.float32)
    if vector.ndim != 1 or vector.size != EXPECTED_EMBEDDING_DIMENSION:
        raise ValueError(f"Expected {EXPECTED_EMBEDDING_DIMENSION} dimensions; got {vector.shape}")
    if not np.isfinite(vector).all():
        raise ValueError("Embedding contains NaN or Inf")
    norm = float(np.linalg.norm(vector))
    if not np.isclose(norm, 1.0, atol=5e-4):
        raise ValueError(f"Unexpected embedding norm: {norm}")
    return vector.tolist()


def embed_group(group):
    results = []
    for start in range(0, len(group), MODAL_REQUEST_BATCH_SIZE):
        request_chunks = group[start:start + MODAL_REQUEST_BATCH_SIZE]
        vectors = embedding_service.embed_texts.remote(
            [c["text"] for c in request_chunks],
            input_type="document",
            batch_size=MODEL_INFERENCE_BATCH_SIZE,
        )
        if len(vectors) != len(request_chunks):
            raise RuntimeError("Modal returned a different number of vectors than inputs")
        for chunk, vector in zip(request_chunks, vectors):
            results.append({**chunk, "embedding": validate_embedding(vector)})
    return results


embedded_by_id = load_embedding_checkpoint(EMBEDDED_FILE)
print("Already embedded:", len(embedded_by_id))
print("Remaining:", sum(c["id"] not in embedded_by_id for c in chunks))

Already embedded: 3464
Remaining: 0


In [5]:
remaining = [c for c in chunks if c["id"] not in embedded_by_id]
total_batches = math.ceil(len(remaining) / CHUNKS_PER_BATCH)

with EMBEDDED_FILE.open("a", encoding="utf-8") as checkpoint:
    for batch_no, start in enumerate(range(0, len(remaining), CHUNKS_PER_BATCH), 1):
        batch = remaining[start:start + CHUNKS_PER_BATCH]
        embedded_batch = embed_group(batch)

        for record in embedded_batch:
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\\n")
        checkpoint.flush()
        os.fsync(checkpoint.fileno())

        embedded_by_id.update({r["id"]: r for r in embedded_batch})
        print(f"Checkpoint {batch_no}/{total_batches}: {len(embedded_by_id)}/{len(chunks)}")

print("Embedding stage complete.")

Embedding stage complete.


In [6]:
embedded_by_id = load_embedding_checkpoint(EMBEDDED_FILE)
missing = ids - embedded_by_id.keys()
unexpected = embedded_by_id.keys() - ids

if missing:
    raise RuntimeError(f"{len(missing)} chunks still lack embeddings")
if unexpected:
    raise RuntimeError(f"{len(unexpected)} checkpoint IDs are not in source data")

for chunk_id, record in embedded_by_id.items():
    validate_embedding(record["embedding"])

print(f"Validated {len(embedded_by_id):,} embeddings; dimension={EXPECTED_EMBEDDING_DIMENSION}")

Validated 3,464 embeddings; dimension=4096


In [7]:
# Connect to the already deployed Modal Chroma service.

VectorStoreRemote = modal.Function.from_name(
    "ace-rag",
    "upsert_chunks",
)

CollectionInfoRemote = modal.Function.from_name(
    "ace-rag",
    "collection_info",
)

print("Connected to the remote Chroma service.")

Connected to the remote Chroma service.


In [8]:
!uv run modal deploy vector_store.py

- Creating objects...
| Creating objects...
└── - Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_store.py: Uploaded 
    0/1 files
/ Creating objects...
└── \ Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_store.py: Uploaded 
    0/1 files
| Creating objects...
└── / Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_store.py: Uploaded 
    0/1 files
Building image im-MuL7rXP0ePN9QfhjepG8Dk
/ Creating objects...
└── \ Creating mount c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_st

=> Step 0: FROM base
/ Creating objects...
└── \ Creating mount c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_st

=> Step 1: RUN python -m pip install chromadb
/ Creating objects...
└── \ Creating mount c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_st
/ Creating objects...
└── \ Creating mount 
    c:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\vector_store.py: 
    Finalizing index of 1 files
\

In [9]:
collection_status = CollectionInfoRemote.remote()

print("Collection:", collection_status["collection"])
print("Current records:", collection_status["count"])
print("Persist directory:", collection_status["persist_directory"])

Collection: ace_documentation
Current records: 0
Persist directory: /data/chroma


In [10]:
embedded_records = load_embedding_checkpoint(EMBEDDED_FILE)

print("Embedding records loaded:", len(embedded_records))

Embedding records loaded: 3464


In [11]:
embedded_records_list = list(embedded_records.values())

print("Records ready for Chroma:", len(embedded_records_list))

Records ready for Chroma: 3464


In [12]:
def upsert_embedding_batch(records):
    chroma_chunks = []
    embeddings = []

    for record in records:
        chroma_chunks.append({
            "id": record["id"],
            "text": record["text"],
            "metadata": record["metadata"],
        })

        embeddings.append(record["embedding"])

    result = VectorStoreRemote.remote(
        chroma_chunks,
        embeddings,
    )

    return result

In [13]:
CHROMA_BATCH_SIZE = 100

total_records = len(embedded_records_list)

total_batches = (
    total_records + CHROMA_BATCH_SIZE - 1
) // CHROMA_BATCH_SIZE

for batch_index, start in enumerate(
    range(0, total_records, CHROMA_BATCH_SIZE),
    start=1,
):
    batch = embedded_records_list[
        start:start + CHROMA_BATCH_SIZE
    ]

    result = upsert_embedding_batch(batch)

    print(
        f"Batch {batch_index}/{total_batches}: "
        f"upserted {result['upserted']} records | "
        f"Chroma collection count: {result['collection_count']}"
    )

print("ChromaDB indexing completed.")

Batch 1/35: upserted 100 records | Chroma collection count: 100
Batch 2/35: upserted 100 records | Chroma collection count: 200
Batch 3/35: upserted 100 records | Chroma collection count: 300
Batch 4/35: upserted 100 records | Chroma collection count: 400
Batch 5/35: upserted 100 records | Chroma collection count: 500
Batch 6/35: upserted 100 records | Chroma collection count: 600
Batch 7/35: upserted 100 records | Chroma collection count: 700
Batch 8/35: upserted 100 records | Chroma collection count: 800
Batch 9/35: upserted 100 records | Chroma collection count: 900
Batch 10/35: upserted 100 records | Chroma collection count: 1000
Batch 11/35: upserted 100 records | Chroma collection count: 1100
Batch 12/35: upserted 100 records | Chroma collection count: 1200
Batch 13/35: upserted 100 records | Chroma collection count: 1300
Batch 14/35: upserted 100 records | Chroma collection count: 1400
Batch 15/35: upserted 100 records | Chroma collection count: 1500
Batch 16/35: upserted 100 re

In [14]:
final_status = CollectionInfoRemote.remote()

expected_count = len(embedded_records_list)
actual_count = final_status["count"]

print("Expected records:", expected_count)
print("Chroma records:", actual_count)
print("Collection:", final_status["collection"])

assert actual_count == expected_count, (
    f"Count mismatch: expected {expected_count}, "
    f"found {actual_count}"
)

print("SUCCESS: All embedding records are stored in ChromaDB.")

Expected records: 3464
Chroma records: 3464
Collection: ace_documentation
SUCCESS: All embedding records are stored in ChromaDB.


In [15]:
!uv run modal volume list

┌────────────────┬──────────────────────────────────────────┬────────────┐
│ Name           │ Created at                               │ Created by │
├────────────────┼──────────────────────────────────────────┼────────────┤
│ ace-rag-chroma │ 2026-11-04 13:51 W. Europe Standard Time │ emil-qsen  │
│ hf-hub-cache   │ 2026-09-03 18:50 W. Europe Standard Time │ emil-qsen  │
└────────────────┴──────────────────────────────────────────┴────────────┘


In [30]:
import modal

UploadChromaRemote = modal.Function.from_name(
    "ace-rag",
    "upload_chroma_to_huggingface",
)

upload_result = UploadChromaRemote.remote()

print(upload_result)

{'status': 'success', 'repository': 'emitev/ace-rag-data', 'path': 'chroma/', 'commit': 'bef761d5cb1403416029835a4c6d3b131a12bcb9'}
